In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
INVESTIGASI KATALOG BMKG DAN USGS (FIXED)
Standarisasi kolom, analisis statistik, dan deteksi duplikat.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

BMKG_PATH = "/Volumes/Extreme SSD/katalog/BMKG_Earthquake_Catalog.csv"
USGS_PATH = "/Volumes/Extreme SSD/katalog/katalog_usgs_earthquake_only.csv"
OUTPUT_DIR = "investigasi_katalog_bmkg_usgs"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# =============================================
# 2. BACA KATALOG BMKG
# =============================================

print("="*70)
print("📂  MEMBACA KATALOG BMKG")
print("="*70)

df_bmkg = pd.read_csv(BMKG_PATH)
df_bmkg.columns = df_bmkg.columns.str.strip()
print(f"✅ Total baris: {len(df_bmkg):,}")
print(f"✅ Kolom: {df_bmkg.columns.tolist()}")

# Konversi tanggal BMKG
def parse_bmkg_date(row):
    try:
        date_str = str(row['Date']).strip()
        time_str = str(row['Time (UTC)']).strip()
        if date_str == 'nan' or time_str == 'nan' or not date_str or not time_str:
            return None
        datetime_str = f"{date_str} {time_str}"
        return pd.to_datetime(datetime_str, format='%d-%b-%Y %H:%M:%S')
    except:
        return None

df_bmkg['datetime'] = df_bmkg.apply(parse_bmkg_date, axis=1)
df_bmkg = df_bmkg.dropna(subset=['datetime'])
df_bmkg['year'] = df_bmkg['datetime'].dt.year

print(f"✅ Setelah parsing datetime: {len(df_bmkg):,} event")
print(f"✅ Rentang tahun: {df_bmkg['year'].min()} - {df_bmkg['year'].max()}")

# Statistik magnitudo
print(f"\n📊 Magnitudo:")
print(f"  Min: {df_bmkg['Magnitude'].min():.2f}")
print(f"  Max: {df_bmkg['Magnitude'].max():.2f}")
print(f"  Mean: {df_bmkg['Magnitude'].mean():.2f}")
print(f"  NaN: {df_bmkg['Magnitude'].isna().sum()}")

# Statistik kedalaman
print(f"\n📊 Kedalaman:")
print(f"  Min: {df_bmkg['Depth (km)'].min():.2f} km")
print(f"  Max: {df_bmkg['Depth (km)'].max():.2f} km")
print(f"  Mean: {df_bmkg['Depth (km)'].mean():.2f} km")
print(f"  NaN: {df_bmkg['Depth (km)'].isna().sum()}")

# Distribusi sumber
print(f"\n📡 Sumber data:")
print(df_bmkg['Source'].value_counts())

# =============================================
# 3. BACA KATALOG USGS (DIPERBAIKI)
# =============================================

print("\n" + "="*70)
print("📂  MEMBACA KATALOG USGS")
print("="*70)

df_usgs = pd.read_csv(USGS_PATH)
df_usgs.columns = df_usgs.columns.str.strip()
print(f"✅ Total baris: {len(df_usgs):,}")
print(f"✅ Kolom: {df_usgs.columns.tolist()}")

# --- PERBAIKAN: Konversi datetime dengan format='mixed' ---
try:
    df_usgs['datetime'] = pd.to_datetime(df_usgs['time'], utc=True, format='mixed')
except Exception as e:
    print(f"⚠️  Format 'mixed' gagal, mencoba 'ISO8601'...")
    df_usgs['datetime'] = pd.to_datetime(df_usgs['time'], utc=True, format='ISO8601')

df_usgs['year'] = df_usgs['datetime'].dt.year
print(f"✅ Rentang tahun: {df_usgs['year'].min()} - {df_usgs['year'].max()}")

# Statistik magnitudo
print(f"\n📊 Magnitudo:")
print(f"  Min: {df_usgs['mag'].min():.2f}")
print(f"  Max: {df_usgs['mag'].max():.2f}")
print(f"  Mean: {df_usgs['mag'].mean():.2f}")
print(f"  NaN: {df_usgs['mag'].isna().sum()}")

# Statistik kedalaman
print(f"\n📊 Kedalaman:")
print(f"  Min: {df_usgs['depth'].min():.2f} km")
print(f"  Max: {df_usgs['depth'].max():.2f} km")
print(f"  Mean: {df_usgs['depth'].mean():.2f} km")
print(f"  NaN: {df_usgs['depth'].isna().sum()}")

# =============================================
# 4. KOMPARASI
# =============================================

print("\n" + "="*70)
print("📊  KOMPARASI KATALOG")
print("="*70)

print(f"\nBMKG:")
print(f"  Total event: {len(df_bmkg):,}")
print(f"  Rentang tahun: {df_bmkg['year'].min()} - {df_bmkg['year'].max()}")
print(f"  Rentang magnitudo: {df_bmkg['Magnitude'].min():.2f} - {df_bmkg['Magnitude'].max():.2f}")
print(f"  Rentang kedalaman: {df_bmkg['Depth (km)'].min():.2f} - {df_bmkg['Depth (km)'].max():.2f} km")
print(f"  Sumber: {df_bmkg['Source'].unique().tolist()}")

print(f"\nUSGS:")
print(f"  Total event: {len(df_usgs):,}")
print(f"  Rentang tahun: {df_usgs['year'].min()} - {df_usgs['year'].max()}")
print(f"  Rentang magnitudo: {df_usgs['mag'].min():.2f} - {df_usgs['mag'].max():.2f}")
print(f"  Rentang kedalaman: {df_usgs['depth'].min():.2f} - {df_usgs['depth'].max():.2f} km")
print(f"  Mag type: {df_usgs['magType'].unique().tolist()}")

# =============================================
# 5. STANDARISASI KOLOM
# =============================================

print("\n" + "="*70)
print("🔧  STANDARISASI KOLOM UNTUK HYBRID")
print("="*70)

cols_bmkg = {
    'datetime': 'datetime',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Magnitude': 'magnitude',
    'Depth (km)': 'depth_km',
    'Source': 'source',
    'Event ID': 'event_id'
}

cols_usgs = {
    'datetime': 'datetime',
    'latitude': 'latitude',
    'longitude': 'longitude',
    'mag': 'magnitude',
    'depth': 'depth_km',
    'net': 'source',
    'id': 'event_id'
}

# Pilih kolom yang ada
bmkg_cols = [c for c in cols_bmkg.keys() if c in df_bmkg.columns]
usgs_cols = [c for c in cols_usgs.keys() if c in df_usgs.columns]

df_bmkg_std = df_bmkg[bmkg_cols].rename(columns=cols_bmkg)
df_bmkg_std['source'] = df_bmkg_std['source'].fillna('BMKG')

df_usgs_std = df_usgs[usgs_cols].rename(columns=cols_usgs)
df_usgs_std['source'] = df_usgs_std['source'].fillna('USGS')

print(f"✅ BMKG standar: {len(df_bmkg_std):,} event")
print(f"✅ USGS standar: {len(df_usgs_std):,} event")

# =============================================
# 6. DETEKSI DUPLIKAT
# =============================================

print("\n" + "="*70)
print("🔍  DETEKSI DUPLIKAT (BMKG vs USGS)")
print("="*70)

df_bmkg_std['time_round'] = df_bmkg_std['datetime'].dt.round('1min')
df_usgs_std['time_round'] = df_usgs_std['datetime'].dt.round('1min')

merged = pd.concat([
    df_bmkg_std[['time_round', 'latitude', 'longitude', 'magnitude', 'source']],
    df_usgs_std[['time_round', 'latitude', 'longitude', 'magnitude', 'source']]
])

duplicates = merged.duplicated(subset=['time_round', 'latitude', 'longitude'], keep=False)
dup_count = duplicates.sum()
print(f"✅ Duplikat (toleransi 1 menit): {dup_count:,} ({dup_count/len(merged)*100:.2f}%)")

# =============================================
# 7. SIMPAN HASIL
# =============================================

df_bmkg_std.to_csv(f"{OUTPUT_DIR}/bmkg_standardized.csv", index=False)
df_usgs_std.to_csv(f"{OUTPUT_DIR}/usgs_standardized.csv", index=False)
print(f"\n✅ BMKG standar: {OUTPUT_DIR}/bmkg_standardized.csv")
print(f"✅ USGS standar: {OUTPUT_DIR}/usgs_standardized.csv")

print("\n" + "="*70)
print("📊  RINGKASAN")
print("="*70)
print(f"BMKG total: {len(df_bmkg):,}")
print(f"USGS total: {len(df_usgs):,}")
print(f"Duplikat terdeteksi: {dup_count:,}")
print("="*70)

📂  MEMBACA KATALOG BMKG
✅ Total baris: 217,807
✅ Kolom: ['No', 'Event ID', 'Unnamed: 2', 'Date', 'Time (UTC)', 'Latitude', 'Longitude', 'Magnitude', 'Mag Type', 'Depth (km)', 'Source', 'Source Event ID']
✅ Setelah parsing datetime: 217,807 event
✅ Rentang tahun: 1998 - 2024

📊 Magnitudo:
  Min: 0.20
  Max: 7.90
  Mean: 3.15
  NaN: 0

📊 Kedalaman:
  Min: 1.00 km
  Max: 798.00 km
  Mean: 43.24 km
  NaN: 0

📡 Sumber data:
Source
NC       98979
RC_03    28890
RC_04    17252
RC_09    15754
RC_02    14998
RC_05    13494
RC_07     9455
RC_01     6775
RC_08     4510
RC_10     4416
RC_06     3284
Name: count, dtype: int64

📂  MEMBACA KATALOG USGS
✅ Total baris: 83,045
✅ Kolom: ['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'nst', 'gap', 'dmin', 'rms', 'net', 'id', 'updated', 'place', 'type', 'horizontalError', 'depthError', 'magError', 'magNst', 'status', 'locationSource', 'magSource']
✅ Rentang tahun: 2001 - 2025

📊 Magnitudo:
  Min: 2.50
  Max: 9.10
  Mean: 4.53
  NaN: 0

📊 Keda

In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
MENGABUNGKAN KATALOG BMKG DAN USGS MENJADI HYBRID CATALOG (FIXED)
Standarisasi kolom, hapus duplikat, filter untuk unduhan waveform.
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

# Path input
BMKG_PATH = "/Volumes/Extreme SSD/katalog/BMKG_Earthquake_Catalog.csv"
USGS_PATH = "/Volumes/Extreme SSD/katalog/katalog_usgs_earthquake_only.csv"

# Path output
OUTPUT_DIR = '/Volumes/Extreme SSD/katalog'
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# Filter parameter untuk unduhan waveform
MIN_MAGNITUDE = 4.5
MAX_DEPTH_KM = 150
MIN_YEAR = 2004
MAX_YEAR = 2024

# =============================================
# 2. BACA KATALOG BMKG
# =============================================

print("="*70)
print("📂  MEMBACA KATALOG BMKG")
print("="*70)

df_bmkg = pd.read_csv(BMKG_PATH)
df_bmkg.columns = df_bmkg.columns.str.strip()
print(f"✅ Total baris: {len(df_bmkg):,}")

# Parsing datetime BMKG
def parse_bmkg_date(row):
    try:
        date_str = str(row['Date']).strip()
        time_str = str(row['Time (UTC)']).strip()
        if date_str == 'nan' or time_str == 'nan' or not date_str or not time_str:
            return None
        datetime_str = f"{date_str} {time_str}"
        return pd.to_datetime(datetime_str, format='%d-%b-%Y %H:%M:%S', utc=True)
    except:
        return None

df_bmkg['datetime'] = df_bmkg.apply(parse_bmkg_date, axis=1)
df_bmkg = df_bmkg.dropna(subset=['datetime'])
df_bmkg['year'] = df_bmkg['datetime'].dt.year

# Standarisasi kolom BMKG
df_bmkg_std = pd.DataFrame({
    'datetime': df_bmkg['datetime'],
    'latitude': df_bmkg['Latitude'],
    'longitude': df_bmkg['Longitude'],
    'magnitude': df_bmkg['Magnitude'],
    'depth_km': df_bmkg['Depth (km)'],
    'source': df_bmkg['Source'].fillna('BMKG'),
    'event_id': df_bmkg['Event ID'],
    'year': df_bmkg['year']
})

print(f"✅ BMKG standar: {len(df_bmkg_std):,} event")
print(f"   Rentang tahun: {df_bmkg_std['year'].min()} - {df_bmkg_std['year'].max()}")

# =============================================
# 3. BACA KATALOG USGS
# =============================================

print("\n📂  MEMBACA KATALOG USGS")
print("="*70)

df_usgs = pd.read_csv(USGS_PATH)
df_usgs.columns = df_usgs.columns.str.strip()
print(f"✅ Total baris: {len(df_usgs):,}")

# Parsing datetime USGS (format ISO)
try:
    df_usgs['datetime'] = pd.to_datetime(df_usgs['time'], utc=True, format='mixed')
except:
    df_usgs['datetime'] = pd.to_datetime(df_usgs['time'], utc=True, format='ISO8601')

df_usgs['year'] = df_usgs['datetime'].dt.year

# Standarisasi kolom USGS
df_usgs_std = pd.DataFrame({
    'datetime': df_usgs['datetime'],
    'latitude': df_usgs['latitude'],
    'longitude': df_usgs['longitude'],
    'magnitude': df_usgs['mag'],
    'depth_km': df_usgs['depth'],
    'source': df_usgs['net'].fillna('USGS'),
    'event_id': df_usgs['id'],
    'year': df_usgs['year']
})

print(f"✅ USGS standar: {len(df_usgs_std):,} event")
print(f"   Rentang tahun: {df_usgs_std['year'].min()} - {df_usgs_std['year'].max()}")

# =============================================
# 4. GABUNGKAN KEDUA KATALOG
# =============================================

print("\n🔗  MENGGABUNGKAN KATALOG")
print("="*70)

# Gabungkan secara vertikal
df_hybrid = pd.concat([df_bmkg_std, df_usgs_std], ignore_index=True)

# --- PERBAIKAN: Pastikan datetime adalah tipe datetime ---
df_hybrid['datetime'] = pd.to_datetime(df_hybrid['datetime'], utc=True)
df_hybrid['year'] = df_hybrid['datetime'].dt.year

print(f"✅ Sebelum deduplikasi: {len(df_hybrid):,} event")

# =============================================
# 5. HAPUS DUPLIKAT
# =============================================

print("\n🔍  MENGHAPUS DUPLIKAT")
print("="*70)

# Round datetime ke menit untuk deteksi duplikat
df_hybrid['time_round'] = df_hybrid['datetime'].dt.round('1min')

# Tandai duplikat berdasarkan waktu+koordinat
duplicate_mask = df_hybrid.duplicated(
    subset=['time_round', 'latitude', 'longitude'],
    keep=False
)
dup_count = duplicate_mask.sum()
print(f"✅ Duplikat terdeteksi: {dup_count:,} ({dup_count/len(df_hybrid)*100:.2f}%)")

if dup_count > 0:
    # Prioritaskan USGS jika ada duplikat (magnitudo lebih akurat)
    priority_order = {'USGS': 0, 'BMKG': 1}
    df_hybrid['priority'] = df_hybrid['source'].map(priority_order).fillna(2)
    
    # Sort dan drop duplicates, keep first (prioritas tertinggi)
    df_hybrid_sorted = df_hybrid.sort_values('priority')
    df_hybrid_clean = df_hybrid_sorted.drop_duplicates(
        subset=['time_round', 'latitude', 'longitude'],
        keep='first'
    ).drop(columns=['priority'])
    
    # Reset index
    df_hybrid_clean = df_hybrid_clean.reset_index(drop=True)
    print(f"✅ Setelah deduplikasi: {len(df_hybrid_clean):,} event")
    print(f"   (dihapus {len(df_hybrid) - len(df_hybrid_clean):,} duplikat)")
else:
    df_hybrid_clean = df_hybrid

# Hapus kolom bantuan
df_hybrid_clean = df_hybrid_clean.drop(columns=['time_round'])

# =============================================
# 6. TERAPKAN FILTER
# =============================================

print("\n🎯  MENERAPKAN FILTER UNTUK UNDUHAN")
print("="*70)

df_filtered = df_hybrid_clean[
    (df_hybrid_clean['magnitude'] >= MIN_MAGNITUDE) &
    (df_hybrid_clean['depth_km'] <= MAX_DEPTH_KM) &
    (df_hybrid_clean['year'] >= MIN_YEAR) &
    (df_hybrid_clean['year'] <= MAX_YEAR)
].copy()

print(f"✅ Event lolos filter:")
print(f"   M ≥ {MIN_MAGNITUDE}")
print(f"   depth ≤ {MAX_DEPTH_KM} km")
print(f"   tahun {MIN_YEAR} - {MAX_YEAR}")
print(f"   Total: {len(df_filtered):,} event ({len(df_filtered)/len(df_hybrid_clean)*100:.1f}%)")

# =============================================
# 7. STATISTIK HASIL FILTER
# =============================================

print("\n📊  STATISTIK HASIL FILTER")
print("="*70)

print(f"Rentang magnitudo: {df_filtered['magnitude'].min():.2f} - {df_filtered['magnitude'].max():.2f}")
print(f"Rentang kedalaman : {df_filtered['depth_km'].min():.2f} - {df_filtered['depth_km'].max():.2f} km")

print("\nDistribusi sumber:")
print(df_filtered['source'].value_counts())

print("\nDistribusi tahun:")
yearly = df_filtered['year'].value_counts().sort_index()
for year, count in yearly.items():
    print(f"  {year}: {count}")

# =============================================
# 8. SIMPAN HASIL
# =============================================

# Hybrid catalog full (tanpa filter)
df_hybrid_clean.to_csv(f"{OUTPUT_DIR}/hybrid_catalog_full.csv", index=False)
print(f"\n💾  Hybrid catalog (full): {OUTPUT_DIR}/hybrid_catalog_full.csv")
print(f"   Total: {len(df_hybrid_clean):,} event")

# Hybrid catalog filtered (untuk unduhan)
df_filtered.to_csv(f"{OUTPUT_DIR}/hybrid_catalog_filtered.csv", index=False)
print(f"💾  Hybrid catalog (filtered): {OUTPUT_DIR}/hybrid_catalog_filtered.csv")
print(f"   Total: {len(df_filtered):,} event")

# =============================================
# 9. RINGKASAN
# =============================================

print("\n" + "="*70)
print("📊  RINGKASAN PENGGABUNGAN")
print("="*70)
print(f"BMKG total: {len(df_bmkg_std):,}")
print(f"USGS total: {len(df_usgs_std):,}")
print(f"Hybrid total (sebelum filter): {len(df_hybrid_clean):,}")
print(f"Hybrid total (setelah filter): {len(df_filtered):,}")
print(f"Filtered persentase: {len(df_filtered)/len(df_hybrid_clean)*100:.1f}%")
print("="*70)

print("\n✅ Selesai! Katalog siap digunakan untuk unduhan waveform.")

📂  MEMBACA KATALOG BMKG
✅ Total baris: 217,807
✅ BMKG standar: 217,807 event
   Rentang tahun: 1998 - 2024

📂  MEMBACA KATALOG USGS
✅ Total baris: 83,045
✅ USGS standar: 83,045 event
   Rentang tahun: 2001 - 2025

🔗  MENGGABUNGKAN KATALOG
✅ Sebelum deduplikasi: 300,852 event

🔍  MENGHAPUS DUPLIKAT
✅ Duplikat terdeteksi: 32 (0.01%)
✅ Setelah deduplikasi: 300,836 event
   (dihapus 16 duplikat)

🎯  MENERAPKAN FILTER UNTUK UNDUHAN
✅ Event lolos filter:
   M ≥ 4.5
   depth ≤ 150 km
   tahun 2004 - 2024
   Total: 49,842 event (16.6%)

📊  STATISTIK HASIL FILTER
Rentang magnitudo: 4.50 - 9.10
Rentang kedalaman : 0.30 - 150.00 km

Distribusi sumber:
source
us          33731
NC          13686
RC_04         966
RC_02         369
RC_09         229
RC_10         195
RC_08         160
RC_01         154
RC_06         107
RC_05         102
RC_03          79
RC_07          47
iscgem         12
official        4
gcmt            1
Name: count, dtype: int64

Distribusi tahun:
  2004: 1768
  2005: 3613
  2

In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
INVESTIGASI HYBRID CATALOG FILTERED
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

CSV_PATH = "/Volumes/Extreme SSD/katalog/hybrid_catalog_filtered.csv"
OUTPUT_DIR = "investigasi_hybrid_filtered"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

print("="*70)
print("📊 INVESTIGASI HYBRID CATALOG FILTERED")
print("="*70)

df = pd.read_csv(CSV_PATH)
print(f"✅ Total baris: {len(df):,}")
print(f"✅ Kolom: {df.columns.tolist()}")

# Statistik dasar
print("\n📊 STATISTIK:")
print(df.describe())

print("\n📊 MISSING VALUES:")
print(df.isnull().sum())

# Distribusi sumber
print("\n📡 SUMBER DATA:")
print(df['source'].value_counts())

# Distribusi tahun
print("\n📊 DISTRIBUSI TAHUN:")
yearly = df['year'].value_counts().sort_index()
print(yearly)

# Plot distribusi tahun
plt.figure(figsize=(14,5))
yearly.plot(kind='bar', color='steelblue')
plt.title('Distribusi Event per Tahun (Hybrid Filtered)')
plt.xlabel('Tahun')
plt.ylabel('Jumlah Event')
plt.grid(True, alpha=0.3)
plt.savefig(f"{OUTPUT_DIR}/yearly_distribution.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"📊 Grafik tersimpan: {OUTPUT_DIR}/yearly_distribution.png")

# Plot magnitudo
plt.figure(figsize=(12,4))
plt.hist(df['magnitude'], bins=30, color='coral', edgecolor='black', alpha=0.7)
plt.axvline(5.0, color='red', linestyle='--', label='M≥5.0')
plt.xlabel('Magnitudo')
plt.ylabel('Frekuensi')
plt.title('Distribusi Magnitudo')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(f"{OUTPUT_DIR}/magnitude_distribution.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"📊 Grafik magnitudo: {OUTPUT_DIR}/magnitude_distribution.png")

# Plot kedalaman
plt.figure(figsize=(12,4))
plt.hist(df['depth_km'], bins=30, color='forestgreen', edgecolor='black', alpha=0.7)
plt.axvline(100, color='red', linestyle='--', label='depth≤100 km')
plt.xlabel('Kedalaman (km)')
plt.ylabel('Frekuensi')
plt.title('Distribusi Kedalaman')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(f"{OUTPUT_DIR}/depth_distribution.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"📊 Grafik kedalaman: {OUTPUT_DIR}/depth_distribution.png")

# Simpan statistik
df.describe().to_csv(f"{OUTPUT_DIR}/statistics.csv")
print(f"\n✅ Statistik tersimpan: {OUTPUT_DIR}/statistics.csv")
print(f"✅ Total event: {len(df):,}")
print(f"✅ Rentang tahun: {df['year'].min()} - {df['year'].max()}")
print("="*70)

📊 INVESTIGASI HYBRID CATALOG FILTERED
✅ Total baris: 49,842
✅ Kolom: ['datetime', 'latitude', 'longitude', 'magnitude', 'depth_km', 'source', 'event_id', 'year']

📊 STATISTIK:
           latitude     longitude     magnitude      depth_km          year
count  49842.000000  49842.000000  49842.000000  49842.000000  49842.000000
mean      -1.347636    123.413032      4.831747     42.554724   2014.021849
std        5.654245     17.969104      0.374956     35.841815      6.043689
min      -14.960000     87.701600      4.500000      0.300000   2004.000000
25%       -6.139950    107.506000      4.600000     10.000000   2009.000000
50%       -2.242000    126.430000      4.700000     33.000000   2014.000000
75%        2.778750    132.003225      5.000000     58.300000   2019.000000
max       11.194000    158.903000      9.100000    150.000000   2024.000000

📊 MISSING VALUES:
datetime     0
latitude     0
longitude    0
magnitude    0
depth_km     0
source       0
event_id     0
year         0
d